# Практика · CNN цілком

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

У лекції ми зібрали згорткову мережу з блоків і порахували, що в ній де лежить. Тут
кожне з тих чисел ти отримаєш сам — і звіриш ручний розрахунок із тим, що показує
PyTorch.

Що зробимо:

1. **Намалюємо датасет** — шість класів фігур 28×28, формулами, без завантажень.
2. **Зберемо мережу з блоків** `Conv → ReLU → Pool` і навчимо її (одне навчання).
3. **Наскрізна таблиця форм**: порахуємо рукою й звіримо з фактичними `.shape` через хуки.
4. **Розподіл параметрів** по шарах у відсотках — і побачимо ті самі 74.7 % у голові.
5. **Оцінка памʼяті активацій** по шарах — і побачимо, що пік у протилежному кінці.
6. **Дослід із розміром входу**: `Flatten`-мережа падає зі справжньою помилкою, GAP-мережа працює.
7. **Рецептивне поле**: порахуємо за формулою й **виміряємо** перебором пікселів.
8. **Порядок ReLU і пулінгу**: перевіримо на числах, коли він важить, а коли ні.

**Мережа не потрібна:** датасет ми малюємо самі, формулами.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)
rng = np.random.default_rng(42)
torch.set_num_threads(4)

print("torch     :", torch.__version__)
print("numpy     :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Датасет: шість фігур, намальованих формулами

Кожне зображення — квадрат 28×28 із однією фігурою: коло, квадрат, ромб, кільце, хрест
або трикутник. Центр трохи зсунутий, радіус трохи різний, зверху накладено шум. Усе це
рахується з координат пікселя, тому датасет однаковий у всіх, хто запустить зошит із
тим самим зерном.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=2, noise=0.10):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо на кілька пікселів, щоб мережа не завчила одне положення
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(7, 10)

    # відстані кожного пікселя від центра — далі з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_shape_dataset(count, rng, jitter=2):
    """Повертає тензори (count, 1, 28, 28) і (count,) з рівною кількістю класів."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


started = time.time()
train_x, train_y = make_shape_dataset(720, rng)
test_x, test_y = make_shape_dataset(600, rng)
print("згенеровано за %.2f с" % (time.time() - started))
print("навчальна вибірка:", tuple(train_x.shape), "мітки:", tuple(train_y.shape))
print("тестова вибірка  :", tuple(test_x.shape))

Подивимось на фігури очима — інакше далі буде незрозуміло, що саме мережа розрізняє.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 6, figsize=(11, 2))
for kind in range(6):
    # перші шість зображень датасету — саме по одному на кожен клас
    axes[kind].imshow(train_x[kind, 0], cmap="gray")
    axes[kind].set_title(SHAPE_NAMES[kind], fontsize=10)
    axes[kind].axis("off")
plt.tight_layout()
plt.show()
print("шість класів, по", len(train_y) // 6, "прикладів кожного в навчальній вибірці")

## 2 · Мережа з блоків

Блок — це три кроки: згортка шукає прикмети, `ReLU` прибирає відʼємні відгуки, пулінг
удвічі зменшує карту. Опишемо блок один раз функцією й складемо з нього тіло мережі.

Голова — `Flatten` і два повнозвʼязні шари. Число 288 у першому з них ми зараз
порахуємо руками, а поки просто впишемо.

In [ ]:
def conv_block(in_channels, out_channels):
    """Один типовий блок: Conv → ReLU → Pool. Порядок саме такий — див. розділ 2 лекції."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class ShapeNet(nn.Module):
    """Три блоки з подвоєнням каналів і голова з Flatten."""

    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            conv_block(1, 8),      # 1×28×28  →  8×14×14
            conv_block(8, 16),     # 8×14×14  → 16×7×7
            conv_block(16, 32),    # 16×7×7   → 32×3×3
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 3 * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 6),
        )

    def forward(self, x):
        return self.head(self.body(x))


torch.manual_seed(0)
net = ShapeNet()
print(net)
print()
print("параметрів усього:", sum(p.numel() for p in net.parameters()))

## 3 · Наскрізна таблиця форм: спершу рукою

Тепер найважливіша навичка теми. Порахуємо форму на виході кожного шару **без PyTorch**,
за двома правилами з лекції:

- згортка 3×3 з `padding=1` і `stride=1` розмір **не міняє**;
- пулінг 2×2 з кроком 2 **ділить розмір на два**, дробова частина відкидається.

Функція нижче — це буквально ці два правила, записані кодом.

In [ ]:
def shapes_by_hand(input_side=28):
    """Рахує форми наскрізь за правилами з лекції. Повертає список (назва, канали, сторона)."""
    side = input_side
    channels = 1
    steps = [("вхід", channels, side)]

    # тіло: три блоки з подвоєнням каналів
    for out_channels in (8, 16, 32):
        # згортка 3×3 з padding=1 лишає сторону такою ж, міняє лише кількість каналів
        channels = out_channels
        steps.append(("Conv %d→%d" % (steps[-1][1], channels), channels, side))
        steps.append(("ReLU", channels, side))
        # пулінг 2×2 з кроком 2 ділить сторону навпіл, дробову частину відкидаємо
        side = side // 2
        steps.append(("MaxPool 2", channels, side))

    flat = channels * side * side
    steps.append(("Flatten", flat, 1))
    steps.append(("Linear %d→64" % flat, 64, 1))
    steps.append(("ReLU", 64, 1))
    steps.append(("Linear 64→6", 6, 1))
    return steps


hand_shapes = shapes_by_hand(28)

print("%-16s %-14s %10s" % ("шар", "вихід", "чисел"))
print("-" * 44)
for name, channels, side in hand_shapes:
    shape_text = "%d×%d×%d" % (channels, side, side) if side > 1 else str(channels)
    print("%-16s %-14s %10d" % (name, shape_text, channels * side * side))

## 4 · Та сама таблиця, але від PyTorch

Тепер перевіримо ручний рахунок. Найпростіший чесний спосіб — **хуки**: попросимо кожен
шар повідомити форму свого виходу, коли крізь нього проходить зображення. Хук
(`forward hook`) — це функція, яку PyTorch викликає після кожного шару.

In [ ]:
actual_shapes = []


def remember_shape(module, module_input, module_output):
    """Хук: записує імʼя шару й форму того, що він видав."""
    actual_shapes.append((type(module).__name__, tuple(module_output.shape)))


# вішаємо хук на кожен «листовий» шар — тобто на той, усередині якого вже немає інших
handles = []
for module in net.modules():
    if len(list(module.children())) == 0:
        handles.append(module.register_forward_hook(remember_shape))

with torch.no_grad():
    net(train_x[:1])                  # одне зображення крізь усю мережу

for handle in handles:                # хуки треба знімати, інакше вони спрацюють і далі
    handle.remove()

print("%-14s %-22s %10s" % ("шар", "фактична форма", "чисел"))
print("-" * 50)
for name, shape in actual_shapes:
    numbers = 1
    for dimension in shape:
        numbers *= dimension
    print("%-14s %-22s %10d" % (name, shape, numbers))

Тепер головне: **звірка**. Ручний рахунок дав список форм, хуки дали свій — вони мусять
збігтися до останнього числа. Порівнюємо кількість чисел на виході кожного шару.

In [ ]:
hand_counts = [channels * side * side for _, channels, side in hand_shapes[1:]]   # без входу
actual_counts = []
for _, shape in actual_shapes:
    numbers = 1
    for dimension in shape:
        numbers *= dimension
    actual_counts.append(numbers)

print("%-16s %12s %12s   %s" % ("шар", "рукою", "PyTorch", "збіг"))
print("-" * 52)
for i, (name, _, _) in enumerate(hand_shapes[1:]):
    same = "так" if hand_counts[i] == actual_counts[i] else "НІ"
    print("%-16s %12d %12d   %s" % (name, hand_counts[i], actual_counts[i], same))

assert hand_counts == actual_counts, "ручний рахунок форм розійшовся з фактичним!"
print("\n✅ усі форми збіглися: ручний рахунок і PyTorch дають те саме")

## 5 · Де живуть параметри

Тепер порахуємо ваги за формулами з лекції:

- згортка: `C_вх × C_вих × k × k + C_вих`;
- повнозвʼязний шар: `вхід × вихід + вихід`.

І одразу звіримо з `sum(p.numel())` — це та сама перевірка «наше проти бібліотечного»,
тільки для параметрів, а не для форм.

In [ ]:
def conv_params(in_channels, out_channels, kernel=3):
    """Ваг у згортці: ядро повторюється для кожної пари каналів, плюс зсув на вихід."""
    return in_channels * out_channels * kernel * kernel + out_channels


def linear_params(in_features, out_features):
    """Ваг у повнозвʼязному шарі: окрема вага на кожну пару чисел, плюс зсув на вихід."""
    return in_features * out_features + out_features


by_hand = [
    ("Conv 1→8",       conv_params(1, 8)),
    ("Conv 8→16",      conv_params(8, 16)),
    ("Conv 16→32",     conv_params(16, 32)),
    ("Linear 288→64",  linear_params(288, 64)),
    ("Linear 64→6",    linear_params(64, 6)),
]

# те саме від PyTorch: беремо кожен шар, у якого взагалі є параметри
from_torch = []
for name, module in net.named_modules():
    count = sum(p.numel() for p in module.parameters(recurse=False))
    if count:
        from_torch.append((name, count))

total = sum(count for _, count in by_hand)
print("%-16s %10s %10s %8s" % ("шар", "рукою", "PyTorch", "частка"))
print("-" * 48)
for (hand_name, hand_count), (torch_name, torch_count) in zip(by_hand, from_torch):
    print("%-16s %10d %10d %7.1f %%"
          % (hand_name, hand_count, torch_count, 100 * hand_count / total))
print("-" * 48)
print("%-16s %10d %10d" % ("усього", total, sum(p.numel() for p in net.parameters())))

assert [c for _, c in by_hand] == [c for _, c in from_torch], "параметри розійшлися!"
print("\n✅ ручний рахунок параметрів збігся з sum(p.numel()) для кожного шару")

Прочитай колонку «частка» уважно — це і є головне відкриття теми. Тепер зведемо її в
два числа: скільки ваг у тілі (три згортки) і скільки в голові.

In [ ]:
conv_total = sum(count for name, count in by_hand if name.startswith("Conv"))
head_total = sum(count for name, count in by_hand if name.startswith("Linear"))
first_linear = by_hand[3][1]

print("усі три згортки : %6d  — %.1f %%" % (conv_total, 100 * conv_total / total))
print("уся голова      : %6d  — %.1f %%" % (head_total, 100 * head_total / total))
print("перший Linear   : %6d  — %.1f %%" % (first_linear, 100 * first_linear / total))
print()
print("Три чверті ваг мережі сидять в одному шарі, який не бачить зображення,")
print("а лише множить розпластаний стовпчик на матрицю.")

## 6 · Де живе памʼять

А тепер той самий розріз, але для **активацій** — чисел, які кожен шар видає й які
зворотне поширення мусить памʼятати. Порахуємо їх у кілобайтах при `float32` (4 байти
на число) і поставимо поруч із вагами того самого шару.

In [ ]:
BYTES_PER_NUMBER = 4          # float32

# беремо форми з хуків і залишаємо тільки шари, які справді щось міняють
rows = []
weights_by_layer = {"Conv2d": [80, 1168, 4640], "Linear": [18496, 390]}
conv_seen, linear_seen = 0, 0
for name, shape in actual_shapes:
    numbers = 1
    for dimension in shape:
        numbers *= dimension
    if name == "Conv2d":
        weight = weights_by_layer["Conv2d"][conv_seen]; conv_seen += 1
    elif name == "Linear":
        weight = weights_by_layer["Linear"][linear_seen]; linear_seen += 1
    else:
        weight = 0
    # ReLU і Flatten не створюють нової карти — це той самий тензор, тож памʼяті не додають
    if name not in ("ReLU", "Flatten"):
        rows.append((name, numbers, weight))

print("%-12s %10s %12s %10s" % ("шар", "чисел", "активації", "ваг"))
print("-" * 48)
for name, numbers, weight in rows:
    print("%-12s %10d %9.1f КБ %10d"
          % (name, numbers, numbers * BYTES_PER_NUMBER / 1024, weight))

peak_activation = max(rows, key=lambda row: row[1])
peak_weights = max(rows, key=lambda row: row[2])
print("-" * 48)
print("пік активацій:", peak_activation[0], "—", peak_activation[1], "чисел,",
      peak_activation[2], "ваг")
print("пік ваг      :", peak_weights[0], "—", peak_weights[1], "чисел,",
      peak_weights[2], "ваг")

Два максимуми стоять у **різних кінцях** мережі. Тепер порахуємо, що це означає для
відеокарти: ваги від батча не залежать взагалі, активації ростуть пропорційно.

In [ ]:
activations_per_image = sum(numbers for _, numbers, _ in rows)
weights_megabytes = total * BYTES_PER_NUMBER / 1024 / 1024

print("активацій на одне зображення:", activations_per_image, "чисел")
print("ваги моделі: %.2f МБ (не залежать від батча)\n" % weights_megabytes)
print("%8s %14s" % ("батч", "активації"))
for batch in (1, 8, 64, 256):
    megabytes = activations_per_image * batch * BYTES_PER_NUMBER / 1024 / 1024
    print("%8d %11.2f МБ" % (batch, megabytes))

## 7 · Навчання

Одне навчання на весь зошит — шість епох на 720 прикладах. Цього досить, щоб мережа
навчилась розрізняти шість фігур; нам тут важлива не рекордна точність, а те, що зібрана
з блоків конструкція справді працює.

In [ ]:
def accuracy(model, images, labels):
    """Частка правильних відповідей; градієнти тут не потрібні, тому no_grad."""
    with torch.no_grad():
        predicted = model(images).argmax(dim=1)
    return (predicted == labels).float().mean().item()


torch.manual_seed(0)
optimizer = torch.optim.Adam(net.parameters(), lr=3e-3)
loss_function = nn.CrossEntropyLoss()
BATCH = 32

started = time.time()
for epoch in range(6):
    order = torch.randperm(len(train_x))          # перемішуємо порядок кожної епохи
    for start in range(0, len(train_x), BATCH):
        batch_index = order[start:start + BATCH]
        optimizer.zero_grad()
        loss = loss_function(net(train_x[batch_index]), train_y[batch_index])
        loss.backward()
        optimizer.step()
    print("епоха %d  втрата %.3f  точність на тесті %.3f"
          % (epoch + 1, loss.item(), accuracy(net, test_x, test_y)))

print("\nнавчання зайняло %.1f с" % (time.time() - started))

## 8 · Дослід із розміром входу

Тепер найцікавіший дослід зошита. Подамо на навчену мережу зображення **іншого розміру**
— 40×40 замість 28×28 — і подивимось на справжню помилку PyTorch. Клітинка нижче
**навмисно падає**: це і є результат, який треба побачити на власні очі.

In [ ]:
bigger_image = torch.zeros(1, 1, 40, 40)

with torch.no_grad():
    body_output = net.body(bigger_image)
print("тіло відпрацювало нормально, карта на виході:", tuple(body_output.shape))
print("Flatten дасть", body_output.numel(), "чисел, а Linear чекає 288")
print("\nа тепер той самий тензор крізь усю мережу:")
net(bigger_image)

Помилка каже все прямо: `mat1 and mat2 shapes cannot be multiplied (1x800 and 288x64)`.
Згортки відпрацювали бездоганно — упав саме повнозвʼязний шар, бо форма його матриці
зафіксована при створенні мережі.

Тепер та сама мережа, але з головою на глобальному усередненні.

In [ ]:
class ShapeNetGAP(nn.Module):
    """Те саме тіло, але голова — глобальне усереднення замість Flatten."""

    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(conv_block(1, 8), conv_block(8, 16), conv_block(16, 32))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),      # від кожного каналу лишається одне число
            nn.Flatten(),
            nn.Linear(32, 6),
        )

    def forward(self, x):
        return self.head(self.body(x))


torch.manual_seed(0)
gap_net = ShapeNetGAP()

print("%8s %14s %14s %10s" % ("сторона", "карта тіла", "у Linear", "вихід"))
print("-" * 50)
for side in (20, 28, 40, 64):
    sample = torch.zeros(1, 1, side, side)
    with torch.no_grad():
        body_map = gap_net.body(sample)
        result = gap_net(sample)
    print("%8d %14s %14d %10s"
          % (side, tuple(body_map.shape)[1:], body_map.shape[1], tuple(result.shape)))

print()
print("параметрів у Flatten-мережі:", sum(p.numel() for p in net.parameters()))
print("параметрів у GAP-мережі    :", sum(p.numel() for p in gap_net.parameters()))

Колонка «у Linear» не змінюється: скільки б клітинок не було в каналі, середнє від них —
одне число, а каналів завжди 32. Саме тому GAP-мережа приймає будь-який розмір входу.
Заразом вона схудла вчетверо.

## 9 · Рецептивне поле: формула проти виміру

Рецептивне поле рахується двома числами наскрізь:

- `r_нове = r_старе + (k − 1) × j_старе` — сторона квадрата на вхідному зображенні;
- `j_нове = j_старе × s` — на скільки вхідних пікселів зсувається вікно між сусідами.

Спершу формула.

In [ ]:
def receptive_field(layers):
    """Наростаючий рахунок r і j по шарах. layers — список (назва, ядро, крок)."""
    r, j = 1, 1
    table = [("вхід", r, j)]
    for name, kernel, stride in layers:
        r = r + (kernel - 1) * j
        j = j * stride
        table.append((name, r, j))
    return table


body_layers = [
    ("Conv 3×3", 3, 1), ("MaxPool 2", 2, 2),
    ("Conv 3×3", 3, 1), ("MaxPool 2", 2, 2),
    ("Conv 3×3", 3, 1),
]

print("%-12s %6s %6s" % ("шар", "r", "j"))
print("-" * 26)
for name, r, j in receptive_field(body_layers):
    print("%-12s %6d %6d" % (name, r, j))

field_by_formula = receptive_field(body_layers)[-1][1]
print("\nодин нейрон останньої згортки бачить квадрат %d×%d на вході"
      % (field_by_formula, field_by_formula))

А тепер **вимір**. Візьмемо один нейрон останньої згортки й перевіримо перебором, які
саме вхідні пікселі на нього впливають: зсунемо кожен піксель по черзі й подивимось, чи
змінився вихід.

Одна тонкість. Рецептивне поле — властивість **геометрії** мережі, а не її ваг: воно не
залежить від того, які числа вивчились. Але `ReLU` обнуляє частину шляхів, і через
випадково мертвий нейрон вимір показав би менше, ніж є насправді. Тому міряємо на копії
тіла **без нелінійності** — форми й кроки в неї ті самі.

In [ ]:
# те саме тіло, але без ReLU: геометрія та сама, жоден шлях не гине на нулі
geometry_only = nn.Sequential(
    net.body[0][0], nn.MaxPool2d(2),
    net.body[1][0], nn.MaxPool2d(2),
    net.body[2][0],
)

base_image = torch.randn(1, 1, 28, 28)

# збираємо батч: перше зображення без змін, далі 784 копії, у кожній зсунуто один піксель
probes = base_image.repeat(28 * 28 + 1, 1, 1, 1)
for pixel in range(28 * 28):
    probes[pixel + 1, 0, pixel // 28, pixel % 28] += 1.0

with torch.no_grad():
    outputs = geometry_only(probes)

# дивимось на центральний нейрон карти 32×7×7: канал 0, позиція (3, 3)
watched = outputs[:, 0, 3, 3]
changed = (watched[1:] - watched[0]).abs() > 1e-6
mask = changed.reshape(28, 28)

rows_touched = mask.any(dim=1).nonzero().flatten()
cols_touched = mask.any(dim=0).nonzero().flatten()
height = int(rows_touched[-1] - rows_touched[0] + 1)
width = int(cols_touched[-1] - cols_touched[0] + 1)

print("пікселів, які впливають на цей нейрон:", int(mask.sum()))
print("вони лежать у рядках %d..%d і стовпцях %d..%d"
      % (rows_touched[0], rows_touched[-1], cols_touched[0], cols_touched[-1]))
print("тобто квадрат %d×%d" % (height, width))
print("формула передбачала  %d×%d" % (field_by_formula, field_by_formula))

assert height == field_by_formula and width == field_by_formula, \
    "вимір рецептивного поля розійшовся з формулою!"
print("\n✅ формула й вимір збіглися")

Порівняємо це з розміром фігури. Радіус у нашому генераторі — від 7 до 9, тобто діаметр
від 14 до 18 пікселів. Рецептивне поле щойно-щойно покриває фігуру — і саме тому мережа
взагалі здатна відрізнити коло від кільця.

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(train_x[0, 0], cmap="gray")
# малюємо квадрат рецептивного поля навколо тієї самої точки, яку міряли
left, top = int(cols_touched[0]), int(rows_touched[0])
plt.gca().add_patch(plt.Rectangle((left - 0.5, top - 0.5), width, height,
                                  fill=False, edgecolor="crimson", linewidth=2))
plt.title("поле %d×%d проти фігури діаметром 14-18" % (height, width), fontsize=10)
plt.axis("off")
plt.show()
print("квадрат %d×%d на зображенні 28×28 — це %.0f %% його площі"
      % (height, width, 100 * height * width / (28 * 28)))

## 10 · Порядок ReLU і пулінгу

Останній дослід. У лекції ми стверджували: з max-пулінгом порядок `ReLU` і пулінгу не
важить, а з avg-пулінгом — важить. Перевіримо це на тому самому шматку карти ознак, який
показував інтерактив.

In [ ]:
feature_map = torch.tensor([[[
    [ 4., -3.,  1.,  6., -2.,  0.],
    [-5.,  2.,  7., -1.,  3., -4.],
    [ 1.,  8., -6.,  2., -3.,  5.],
    [ 3., -2.,  0.,  9., -7.,  1.],
    [-1.,  5.,  4., -3.,  2.,  6.],
    [ 2., -4.,  3.,  1.,  8., -5.],
]]])

relu = nn.ReLU()
max_pool = nn.MaxPool2d(2)
avg_pool = nn.AvgPool2d(2)

max_first_relu = max_pool(relu(feature_map))
max_first_pool = relu(max_pool(feature_map))
avg_first_relu = avg_pool(relu(feature_map))
avg_first_pool = relu(avg_pool(feature_map))

print("max-пулінг, ReLU спершу:\n", max_first_relu[0, 0].numpy())
print("max-пулінг, пулінг спершу:\n", max_first_pool[0, 0].numpy())
print("однаково:", torch.equal(max_first_relu, max_first_pool))
print()
print("avg-пулінг, ReLU спершу:\n", avg_first_relu[0, 0].numpy())
print("avg-пулінг, пулінг спершу:\n", avg_first_pool[0, 0].numpy())
print("однаково:", torch.equal(avg_first_relu, avg_first_pool))
print("найбільше розходження:",
      (avg_first_relu - avg_first_pool).abs().max().item())

assert torch.equal(max_first_relu, max_first_pool), "з максимумом порядок мав не важити!"
assert not torch.equal(avg_first_relu, avg_first_pool), "із середнім порядок мав важити!"
print("\n✅ з max-пулінгом порядок вільний, з avg-пулінгом — ні")

Причина видна на квадраті `−5, 3, −1, 4`. Спершу `ReLU`, потім середнє:
`(0 + 3 + 0 + 4) / 4 = 1.75`. Спершу середнє: `(−5 + 3 − 1 + 4) / 4 = 0.25`. У другому
випадку відʼємні числа встигли погасити додатні до того, як їх обнулили.

---

## Що далі

Ти зібрав мережу з блоків, порахував рукою кожну її форму й кожну вагу, звірив усе з
PyTorch і виміряв рецептивне поле замість того, щоб повірити формулі. Головні числа
цього зошита:

- **74.7 %** ваг мережі — в одному шарі, першому `Linear`;
- **6 272** активації після першої згортки проти **64** після цього `Linear`;
- **18×18** — рецептивне поле останньої згортки при фігурі діаметром 14-18;
- **вчетверо** менша мережа з головою на глобальному усередненні, яка ще й приймає будь-який
  розмір входу.

Далі — [домашнє завдання](homework.html) на три рівні. Найцікавіший третій: зібрати
класифікатор під бюджет у 10 000 параметрів і **довести розрахунком**, що бюджет
дотримано, ще до того як запускати код.